# GTF to TSS extraction

Extracts one transcription start site (TSS) per gene from a GENCODE GTF file,
producing a bed file usable downstream for TF-gene assignment.

**Input**: an already-unzipped `.gtf` file (plain text).
**Output**: a bed file with columns: chrom, tss_start, tss_end, gene_name, score, strand.


In [3]:
gtf_path = "../reference_data/2026-07-30_raw-gencode-v50-annotation.gtf"       # path to unzipped GTF file
out_path = "../reference_data/2026-07-31_unsorted-gencode-tss-protein-encoding.bed"     # output bed file

## Step 1: Read the GTF line by line

A GTF file has one row per genomic feature (gene, transcript, exon, ...).
We only want rows where the feature type (column 3) is `"gene"` -- this
already covers the gene's full span, so we don't need transcripts or exons.

Lines starting with `#` are header/comment lines and get skipped.


In [ ]:
gene_rows = []
n_total_rows = 0

with open(gtf_path) as f:
    for line in f:
        n_total_rows += 1      #n=n+1
        if line.startswith("#"):
            continue  # skip header/comment lines

        cols = line.rstrip("\n").split("\t")

        # Filter to get only gene rows
        feature_type = cols[2]
        if feature_type != "gene":
            continue  # skip transcript/exon/CDS/etc rows

        # Filter to get only protein-encoding genes ---------------------------------------- new
        if "protein_coding" not in cols[8]: # the long semicolon-separated string
            continue

        gene_rows.append(cols)

print(f"Found {len(gene_rows)} gene rows out of {n_total_rows} total rows")

Found 20107 gene rows out of 11242719 total rows


## Step 2: Pull out the gene name

Each row's last column (attributes) is a semicolon-separated string like:

```
gene_id "ENSG00000223972.5"; gene_type "lncRNA"; gene_name "DDX11L1"; level 2; tag:"overlaps_pseudogene"
```

We want the human-readable `gene_name` value (not the Ensembl ID), since
that's what we'll match against ChIP-seq-derived gene assignments later.


In [ ]:
def extract_gene_name(attributes_str):
    for attr in attributes_str.split(";"):
        attr = attr.strip()
        if attr.startswith("gene_name"):
            return attr.split('"')[1]
    return None

# quick check on the first row
# Note: there are 9 columns (indexed 0 to 8), the last one has 5 parts (semicolon separated)
extract_gene_name(gene_rows[0][8])

'OR4F5'

## Step 3: Compute the TSS, accounting for strand

This is the step most likely to go wrong if rushed.

- **Plus-strand gene**: transcription reads left to right, so the TSS is the
  **start** coordinate (column 4).
- **Minus-strand gene**: transcription reads right to left, so the TSS is the
  **end** coordinate (column 5) -- the biological "start" sits at the
  higher genomic coordinate.

Using the wrong one silently gives every minus-strand gene a TSS at the
wrong end of the gene.

We also convert from the GTF's 1-based coordinates to bed's 0-based
coordinates (subtract 1), to stay consistent with narrowPeak files, which
are already 0-based.


In [7]:
def compute_tss(start, end, strand):
    start = int(start)
    end = int(end)
    if strand == "+":
        tss = start - 1   # GTF 1-based -> bed 0-based
    else:
        tss = end - 1
    return tss

# quick check: one + strand gene, one - strand gene
print(compute_tss(11869, 14409, "+"))   # expect 11868
print(compute_tss(14404, 29570, "-"))   # expect 29569

11868
29569


## Step 4: Build the output rows and write the bed file

Bed format, one row per gene:

```
chrom  tss_start  tss_end  gene_name  score  strand
```


In [12]:
output_rows = []

for cols in gene_rows:
    chrom = cols[0]
    start, end, strand = cols[3], cols[4], cols[6]
    attributes = cols[8]

    gene_name = extract_gene_name(attributes)
    if gene_name is None:
        continue  # skip if no gene_name field present

    tss = compute_tss(start, end, strand)
    tes = compute_tss(end, start, strand)  # transcription end site - clever trick?? ----------------------- new
    output_rows.append(f"{chrom}\t{tss}\t{tes}\t{gene_name}\t.\t{strand}")

with open(out_path, "w") as f_out:
    f_out.write("\n".join(output_rows) + "\n")

print(f"Wrote {len(output_rows)} gene TSS entries to {out_path}")

Wrote 20107 gene TSS entries to ../reference_data/2026-07-31_unsorted-gencode-tss-protein-encoding.bed


## Step 5: Check output


In [13]:
with open(out_path) as f:
    for line in f:
        print(line.strip())


chr1	65418	71584	OR4F5	.	+
chr1	451677	450739	OR4F29	.	-
chr1	686653	685715	OR4F16	.	-
chr1	923894	944589	SAMD11	.	+
chr1	960713	943526	NOC2L	.	-
chr1	960554	965720	KLHL17	.	+
chr1	966436	975864	PLEKHN1	.	+
chr1	982136	975193	PERM1	.	-
chr1	1000824	998961	HES4	.	-
chr1	1001137	1014539	ISG15	.	+
chr1	1020068	1056118	AGRN	.	+
chr1	1074305	1070966	RNF223	.	-
chr1	1116360	1081810	C1orf159	.	-
chr1	1173879	1197939	TTLL10	.	+
chr1	1206813	1203507	TNFRSF18	.	-
chr1	1214152	1211325	TNFRSF4	.	-
chr1	1240963	1216886	SDF4	.	-
chr1	1232236	1235040	B3GALT6	.	+
chr1	1246933	1242445	C1QTNF12	.	-
chr1	1274476	1253906	UBE2J2	.	-
chr1	1280435	1292028	SCNN1D	.	+
chr1	1309608	1292375	ACAP3	.	-
chr1	1308546	1311678	PUSL1	.	+
chr1	1324687	1311584	INTS11	.	-
chr1	1324755	1328896	CPTP	.	+
chr1	1331274	1335313	TAS1R3	.	+
chr1	1349961	1335274	DVL1	.	-
chr1	1361799	1352682	MXRA8	.	-
chr1	1375520	1373709	AURKAIP1	.	-
chr1	1400908	1385708	CCNL2	.	-
chr1	1407305	1401906	MRPL20	.	-
chr1	1421768	1415636	ANKRD65	.	-
c

## Next steps
1. Sort BED file (in bash): `sort -k1,1 -k2,2n unsorted.bed > sorted.bed`
- -k1,1: sort by the 1st column only (chromosome name, e.g. chr1, chr2), alphabetically.
- -k2,2n: within same chromosome, sort by the 2nd column (start position), numerically (n flag — without it, sort treats numbers as text, so "10" would come before "2").
2. Use ML model to filter spurious peaks out of narrowPeaks file
3. Build GRN (from sorted BED and filtered narrowPeaks)
4. Maybe incorporate Hi-C data and rebuild GRN
5. Analysis


In [ ]:
# Check file sizes
n=0
with open('../reference_data/2026-07-30_unsorted-gencode-tss-with-ncRNA.bed') as f:
    for line in f:
        n+=1
print(n) # 78,733

m=0
with open('../reference_data/2026-07-31_unsorted-gencode-tss-protein-encoding.bed') as f:
    for line in f:
        m+=1
print(m) # 20,107


78733
20107
